**Enrollment No.:240840116048**

## **Practical-10** Mini Project: Analyze a real-world dataset, build an interactive dashboard, and present actionable insights with ethicalconsiderations.

**Step 1 : Installation**

In [14]:
!pip install pandas plotly dash

**Step 2 : Import library**

In [16]:
# import libraries
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc, Input, Output

**Step 3 : Load dataset**

In [15]:
# Load Dataset
from google.colab import files

uploaded = files.upload()

Saving hotel_bookings_sample .csv to hotel_bookings_sample  (1).csv


In [4]:
# Display the dataset
df = pd.read_csv("hotel_bookings_sample .csv")
df

,Booking_ID,Booking_Date,Hotel_Type,City,Booking_Channel,Market_Segment,Guests,Nights,Children,Parking_Spaces,ADR,Is_Canceled,Status
0,1001,2025-01-05,City Hotel,Lisbon,Online,Direct,2,3,1.0,0.0,145.0,No,Check-Out
1,1002,2025-01-08,Resort Hotel,Goa,Offline,TA/TO,2,5,0.0,1.0,220.0,No,Check-Out
2,1003,2025-01-12,City Hotel,Mumbai,Online,Direct,1,2,0.0,0.0,NaN,Yes,Canceled
3,1004,2025-01-18,Resort Hotel,Goa,Online,Corporate,2,4,1.0,0.0,260.0,No,Check-Out
4,1005,2025-02-02,City Hotel,Ahmedabad,Offline,TA/TO,3,2,0.0,0.0,95.0,No,Check-Out
5,1006,2025-02-07,City Hotel,Mumbai,Online,Direct,2,1,0.0,0.0,130.0,Yes,Canceled
6,1007,2025-02-11,Resort Hotel,Jaipur,Online,Direct,2,6,NaN,0.0,310.0,No,Check-Out
7,1008,2025-02-17,City Hotel,Ahmedabad,Offline,Corporate,1,3,0.0,0.0,105.0,No,Check-Out
8,1009,2025-03-03,Resort Hotel,Goa,Online,TA/TO,2,4,1.0,1.0,240.0,No,Check-Out
9,1010,2025-03-09,City Hotel,Lisbon,Online,Direct,2,2,0.0,0.0,155.0,No,Check-Out


**Step 4 : Dataset Cleaning**

In [12]:
# 2. Data Cleaning
df = df.drop_duplicates().copy()

df["Booking_Date"] = pd.to_datetime(
    df["Booking_Date"], errors="coerce"
)

for col in ["City", "Booking_Channel", "Market_Segment"]:
    df[col] = df[col].fillna("Unknown")

df["Children"] = df["Children"].fillna(0)
df["Parking_Spaces"] = df["Parking_Spaces"].fillna(0)
df["ADR"] = df["ADR"].fillna(df["ADR"].median())

df["Month"] = df["Booking_Date"].dt.strftime("%b")

df["Cancellation"] = df["Is_Canceled"].map(
    {"Yes": "Cancelled", "No": "Not Cancelled"}
)

**Step 5: Create and Interactive Dashboard**

In [13]:
# 3. Initialize Dash App
app = Dash(__name__)
app.title = "Hotel Booking Dashboard"

# 4. Dropdown Options
def options(column):
    return [
        {"label": "All", "value": "All"}
    ] + [
        {"label": str(x), "value": str(x)}
        for x in sorted(df[column].unique())
    ]

# 5. Dashboard Layout
app.layout = html.Div([
    html.H1(
        "Hotel Booking Analysis Dashboard",
        style={"textAlign": "center"}
    ),

    html.P(
        "Analyze hotel performance and booking trends",
        style={"textAlign": "center"}
    ),

    html.Div([
        html.Div([
            html.Label("Hotel Type"),
            dcc.Dropdown(
                id="hotel",
                options=options("Hotel_Type"),
                value="All",
                clearable=False
            )
        ], style={"width": "32%"}),

        html.Div([
            html.Label("City"),
            dcc.Dropdown(
                id="city",
                options=options("City"),
                value="All",
                clearable=False
            )
        ], style={"width": "32%"}),

        html.Div([
            html.Label("Booking Channel"),
            dcc.Dropdown(
                id="channel",
                options=options("Booking_Channel"),
                value="All",
                clearable=False
            )
        ], style={"width": "32%"})
    ], style={
        "display": "flex",
        "gap": "15px",
        "margin": "20px"
    }),

    html.Div(id="kpi", style={
        "display": "flex",
        "justifyContent": "space-around",
        "flexWrap": "wrap"
    }),

    html.Div([
        dcc.Graph(id="city_chart"),
        dcc.Graph(id="hotel_chart")
    ], style={"display": "grid",
              "gridTemplateColumns": "1fr 1fr"}),

    html.Div([
        dcc.Graph(id="monthly_chart"),
        dcc.Graph(id="channel_chart")
    ], style={"display": "grid",
              "gridTemplateColumns": "1fr 1fr"})
], style={
    "padding": "20px",
    "backgroundColor": "#f4f6f9"
})


# 6. Interactive Dashboard Callback
@app.callback(
    Output("kpi", "children"),
    Output("city_chart", "figure"),
    Output("hotel_chart", "figure"),
    Output("monthly_chart", "figure"),
    Output("channel_chart", "figure"),
    Input("hotel", "value"),
    Input("city", "value"),
    Input("channel", "value")
)
def update_dashboard(hotel, city, channel):

    dff = df.copy()

    if hotel != "All":
        dff = dff[dff["Hotel_Type"] == hotel]

    if city != "All":
        dff = dff[dff["City"] == city]

    if channel != "All":
        dff = dff[dff["Booking_Channel"] == channel]

    total = len(dff)

    cancelled = (dff["Is_Canceled"] == "Yes").sum()

    rate = (cancelled / total * 100) if total else 0

    avg_adr = dff["ADR"].mean() if total else 0

    avg_nights = dff["Nights"].mean() if total else 0

    def card(title, value):
        return html.Div([
            html.H4(title),
            html.H2(value)
        ], style={
            "backgroundColor": "white",
            "padding": "15px",
            "margin": "8px",
            "textAlign": "center",
            "borderRadius": "10px",
            "minWidth": "150px",
            "flex": "1"
        })

    kpis = [
        card("Total Bookings", str(total)),
        card("Cancelled", str(cancelled)),
        card("Cancellation Rate", f"{rate:.1f}%"),
        card("Average ADR", f"{avg_adr:.2f}"),
        card("Average Nights", f"{avg_nights:.1f}")
    ]

    # City-wise bookings
    city_data = dff.groupby("City").size().reset_index(
        name="Bookings"
    )

    fig1 = px.bar(
        city_data,
        x="City",
        y="Bookings",
        title="Bookings by City",
        color="Bookings"
    )

    # Hotel cancellation comparison
    hotel_data = dff.groupby("Hotel_Type").agg(
        Bookings=("Booking_ID", "count"),
        Cancelled=("Is_Canceled",
                   lambda x: (x == "Yes").sum())
    ).reset_index()

    hotel_data["Cancellation Rate"] = (
        hotel_data["Cancelled"] /
        hotel_data["Bookings"] * 100
    )

    fig2 = px.bar(
        hotel_data,
        x="Hotel_Type",
        y="Cancellation Rate",
        title="Cancellation Rate by Hotel Type",
        text_auto=".1f"
    )

    # Monthly booking trends
    month_order = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
    ]

    monthly = (
        dff.groupby("Month").size()
        .reindex(month_order)
        .dropna()
        .reset_index(name="Bookings")
    )

    fig3 = px.line(
        monthly,
        x="Month",
        y="Bookings",
        markers=True,
        title="Monthly Booking Trends"
    )

    # Booking channel analysis
    channel_data = dff["Booking_Channel"].value_counts(
    ).reset_index()

    channel_data.columns = ["Channel", "Bookings"]

    fig4 = px.pie(
        channel_data,
        names="Channel",
        values="Bookings",
        title="Bookings by Channel"
    )

    return kpis, fig1, fig2, fig3, fig4


# 7. Run Dashboard
if __name__ == "__main__":
    app.run(debug=True)


<IPython.core.display.Javascript object>

## **Business Insights and Findings**
1. Booking Performance: Out of 24 unique bookings, 20 were completed and 4 were cancelled.

2. Hotel Type Analysis: City Hotel received 13 bookings, while Resort Hotel received 11 bookings. City Hotel has a higher cancellation rate.

3. City Analysis: Mumbai has the highest cancellation rate (60%), followed by Ahmedabad (25%).

4. Booking Channel: Online booking is the most popular channel, accounting for 62.5% of bookings.

5. ADR Analysis: Resort Hotel has a higher average daily rate (282.73) than City Hotel (128.46).

6. Monthly Trends: Bookings remained consistent, with 4 bookings recorded each month from January to June.

7. Customer Segment: Direct booking is the largest known market segment, with 10 bookings.
## **Actionable Recommendations**
- Improve booking confirmation to reduce cancellations.
- Investigate high cancellation rates in Mumbai.
- Improve online booking services.
- Maintain competitive room pricing.
- Encourage direct bookings and customer loyalty.
## **Ethical Considerations**
- Protect customer privacy.
- Maintain accurate data.
- Avoid customer discrimination.
- Use data responsibly for business decisions.